# Bulk RNA-seq toy data generator

From Claude Science

27 Oct 2026

In [4]:
#!/usr/bin/env Rscript
# =============================================================================
# simulate_toy_counts.R
# Toy RNA-seq RAW count matrix for DESeq2 testing.
#   2000 genes (rows) x 6 samples (cols): 3 control + 3 case
#   ~10% of genes differentially expressed.
#
# Counts are drawn from a negative-binomial model with a mean-dependent
# dispersion trend (the same family DESeq2 assumes), so the matrix is a fair
# test substrate rather than Gaussian noise.
#
# IMPORTANT distinction (see de_expr_floor_quantile below):
#   * "truly DE"            = genes you inject a fold change into (always 10% here)
#   * "significant on DESeq2" = genes DESeq2 recovers at padj < alpha
# At n = 3/group these differ: low-count true-DE genes are underpowered.
# Verified with pydeseq2 on this exact model:
#   - DE across full expression range -> ~7.6% called at padj<0.05 (FDR ~0.04)
#   - DE restricted to expressed genes -> ~10.0% called (sensitivity ~0.97)
# =============================================================================

set.seed(42)

## ---- parameters you may want to change ------------------------------------
n_genes    <- 2000      # rows
n_per_grp  <- 4         # samples per group (-> 6 columns total)
frac_de    <- 0.10      # fraction of genes made truly DE (200 genes)

# Baseline expression: log-normal mean counts -> realistic wide dynamic range.
base_meanlog <- 4.0     # median base mean ~ exp(4) ~= 55 counts
base_sdlog   <- 2.0     # spread across orders of magnitude

# Mean-dependent dispersion (DESeq2 parametric form: a + b/mu) + gene scatter.
asympt_disp  <- 0.02    # asymptotic dispersion at high counts
extra_pois   <- 1.0     # 1/mu term
disp_logsd   <- 0.3     # gene-to-gene scatter around the trend

# Effect sizes for the DE genes (log2 fold change, case vs control).
lfc_sd       <- 1.2     # spread of effects
min_abs_lfc  <- 1.0     # enforce |log2FC| >= 1 (>= 2-fold) so DE is real

# Which genes are ELIGIBLE to be DE, by expression percentile:
#   0.00 -> DE anywhere (10% truly DE; DESeq2 recovers ~7.6% at n=3 -> realistic
#           power gap, low-count DE genes stay non-significant)
#   0.40 -> DE only among the better-expressed ~60% (DESeq2 calls ~10%)
de_expr_floor_quantile <- 0.40

run_deseq2_check <- TRUE # if DESeq2 is installed, report recovery vs truth
alpha            <- 0.05 # padj threshold for the check

## ---- derived --------------------------------------------------------------
n_de  <- round(n_genes * frac_de)
group <- factor(c(rep("control", n_per_grp), rep("case", n_per_grp)),
                levels = c("control", "case"))
n_samp <- length(group)

In [5]:
## ---- 1. baseline mean per gene --------------------------------------------
base_mean <- pmax(rlnorm(n_genes, meanlog = base_meanlog, sdlog = base_sdlog), 1)

## ---- 2. per-gene dispersion (mean-dependent trend + scatter) --------------
disp_trend <- asympt_disp + extra_pois / base_mean
disp       <- disp_trend * exp(rnorm(n_genes, 0, disp_logsd))

## ---- 3. per-sample size factors (sequencing-depth variation) --------------
size_factors <- 2 ^ runif(n_samp, -0.3, 0.3)   # ~0.8x - 1.23x

## ---- 4. assign true log2 fold changes to the DE genes ---------------------
true_lfc <- numeric(n_genes)
eligible <- which(base_mean >= quantile(base_mean, de_expr_floor_quantile))
de_idx   <- sort(sample(eligible, n_de))
mag      <- rnorm(n_de, 0, lfc_sd)
mag      <- sign(mag) * (abs(mag) + min_abs_lfc)   # push off zero, keep sign
true_lfc[de_idx] <- mag

## ---- 5. expected counts: base * size factor * fold change (case only) -----
mu <- outer(base_mean, size_factors)               # genes x samples
fc <- matrix(1, n_genes, n_samp)
fc[, group == "case"] <- 2 ^ true_lfc              # recycled across case cols
mu <- mu * fc

## ---- 6. draw negative-binomial counts  (var = mu + disp*mu^2) -------------
# NB 'size' parameter = 1 / dispersion; recycle per-gene to match column-major mu
size_param <- 1 / disp
counts <- matrix(
  rnbinom(n_genes * n_samp, mu = as.vector(mu), size = rep(size_param, n_samp)),
  nrow = n_genes
)
rownames(counts) <- sprintf("gene%04d", seq_len(n_genes))
colnames(counts) <- c(sprintf("ctrl%d", seq_len(n_per_grp)),
                      sprintf("case%d", seq_len(n_per_grp)))

## ---- 7. ground-truth table and sample sheet -------------------------------
truth <- data.frame(
  gene      = rownames(counts),
  base_mean = round(base_mean, 2),
  dispersion = round(disp, 4),
  true_lfc  = round(true_lfc, 3),
  is_de     = seq_len(n_genes) %in% de_idx,
  stringsAsFactors = FALSE
)
coldata <- data.frame(condition = group, row.names = colnames(counts))

## ---- 8. write outputs -----------------------------------------------------
write.csv(counts,  "toy_counts.csv")                      # raw integer counts
write.csv(truth,   "toy_truth.csv", row.names = FALSE)    # answer key
write.csv(coldata, "toy_coldata.csv")                     # DESeq2 colData

cat(sprintf("Wrote toy_counts.csv  (%d genes x %d samples)\n", n_genes, n_samp))
cat(sprintf("  truly DE: %d genes (%.0f%%)  |  zeros: %.1f%%  |  median count: %.0f\n",
            n_de, 100 * frac_de, 100 * mean(counts == 0), median(counts)))

Wrote toy_counts.csv  (2000 genes x 8 samples)
  truly DE: 200 genes (10%)  |  zeros: 3.2%  |  median count: 51


In [6]:
## ---- 9. optional: round-trip through DESeq2 and report recovery -----------
if (run_deseq2_check && requireNamespace("DESeq2", quietly = TRUE)) {
  suppressPackageStartupMessages(library(DESeq2))
  dds <- DESeqDataSetFromMatrix(counts, coldata, design = ~ condition)
  dds <- DESeq(dds, quiet = TRUE)
  res <- results(dds, contrast = c("condition", "case", "control"))
  called <- !is.na(res$padj) & res$padj < alpha
  tp <- sum(called &  truth$is_de)
  fp <- sum(called & !truth$is_de)
  cat(sprintf("\nDESeq2 @ padj<%.2f: %d called (%.1f%%)  TP=%d/%d  FP=%d  sens=%.2f  FDR=%.2f\n",
              alpha, sum(called), 100 * mean(called), tp, n_de, fp,
              tp / n_de, fp / max(sum(called), 1)))
} else if (run_deseq2_check) {
  cat("\n(DESeq2 not installed - skipped recovery check.\n",
      " BiocManager::install('DESeq2') to enable it.)\n", sep = "")
}

# ---- how to load into DESeq2 yourself -------------------------------------
# counts  <- as.matrix(read.csv("toy_counts.csv",  row.names = 1))
# coldata <- read.csv("toy_coldata.csv", row.names = 1)
# dds <- DESeqDataSetFromMatrix(counts, coldata, design = ~ condition)
# dds <- DESeq(dds); res <- results(dds, contrast=c("condition","case","control"))

converting counts to integer mode




DESeq2 @ padj<0.05: 204 called (10.2%)  TP=196/200  FP=8  sens=0.98  FDR=0.04
